In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

In [ ]:
path = hf_hub_download(repo_id="WAZOBIALABS/nigerian-pidgin-voice-text",filename="wazobia_pidgin_v07_500.csv",repo_type="dataset")

df = pd.read_csv(path)

wazobia_pidgin_v07_500.csv: 0.00B [00:00, ?B/s]

(500, 15)
['entry_id', 'pidgin_text', 'english_gloss', 'sentiment', 'emotion_category', 'emotion_secondary', 'register', 'sarcasm_flag', 'prosody_match', 'annotator_notes', 'topic_domain', 'speaker_gender', 'intensity', 'source_type', 'date_added']


In [ ]:
# Inspect immediately
print(df.shape)
print(df.columns.tolist())

In [27]:
df

,entry_id,pidgin_text,english_gloss,sentiment,emotion_category,emotion_secondary,register,sarcasm_flag,prosody_match,annotator_notes,topic_domain,speaker_gender,intensity,source_type,date_added
0,WZ-T-0001,How you dey?,How are you?,neutral,neutral,NaN,casual,no,matched,Standard Lagos greeting,greeting,neutral,1,original,2026-04-23
1,WZ-T-0002,How body?,How are you feeling?,neutral,neutral,NaN,casual,no,matched,More intimate than how you dey,greeting,neutral,1,original,2026-04-23
2,WZ-T-0003,You don chop?,Have you eaten?,neutral,neutral,NaN,casual,no,matched,Care expression very common,food,neutral,1,original,2026-04-23
3,WZ-T-0004,You don try,Well done — sincere,positive,joy,NaN,casual,no,matched,Sincere warm praise,social,neutral,1,original,2026-04-23
4,WZ-T-0005,You don try oh,You really did well,positive,joy,NaN,casual,no,matched,Oh increases sincerity and warmth,social,neutral,2,original,2026-04-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,WZ-T-0496,"I dey on my grind, watch how e go turn out","I am grinding hard, watch how it will turn out",positive,hustle_energy,pride,street,no,matched,Confident grind energy — the outcome is antici...,motivation,female,4,original,2026-05-17
496,WZ-T-0497,I no get time for this kind person,I have no time for this kind of person,negative,contempt,suspicion,casual,no,matched,Cold contempt — dismissal without anger. The p...,relationship,female,3,original,2026-05-17
497,WZ-T-0498,E no reach my level abeg,"He/she has not reached my level, please",negative,contempt,pride,street,no,matched,Contempt expressed as superiority — the other ...,social,neutral,3,original,2026-05-17
498,WZ-T-0499,Who be you sef?,Who are you even?,negative,contempt,anger,street,no,matched,Sharp contempt question — sef amplifies the di...,conflict,neutral,4,original,2026-05-17


In [10]:
# ─── Label maps (must match your dataset exactly) ──────────────────────────────
 
SENTIMENT_LABELS = {"positive": 0, "negative": 1, "neutral":  2,}
 
EMOTION_LABELS = {"joy":0, "sadness":1,"anger":2,"fear":3,
                    "surprise":4, "disgust":5, "trust":6,"anticipation":7,
                    "love":8,"shame":9, "guilt":10,"pride":11,"envy":12,
                    "gratitude":13, "confusion":14, "neutral":15,}
 
SARCASM_LABELS = {"no":  0, "yes": 1,}
 
PROSODY_LABELS = {"matched":  0, "contradicts": 1,}
 
REGISTER_LABELS = {"casual": 0, "street": 1, "proverbial": 2,}
 
NUM_EMOTION_CLASSES = len(EMOTION_LABELS)

In [9]:
df.duplicated().any()

False

In [8]:
df.isnull().sum()

entry_id               0
pidgin_text            0
english_gloss          0
sentiment              0
emotion_category       0
emotion_secondary    353
register               0
sarcasm_flag           0
prosody_match          0
annotator_notes        0
topic_domain           0
speaker_gender         0
intensity              0
source_type            0
date_added             0
dtype: int64

In [ ]:
# 7. Unique emotion values 
raw_vals = df["emotion_category"].unique()
print(raw_vals)
print("\n Unique emotion_secondary values")
raw_sec = df["emotion_secondary"].dropna().unique()
print(raw_sec)


['neutral' 'joy' 'celebration' 'sarcasm' 'hustle_fatigue' 'anger'
 'prayer_gratitude' 'forming' 'suspicion' 'shock' 'pride' 'market_energy'
 'craving' 'contempt' 'betrayal' 'hustle_energy']

 Unique emotion_secondary values
['betrayal' 'anger' 'shock' 'suspicion' 'hustle_fatigue' 'pride'
 'prayer_gratitude' 'joy' 'celebration' 'craving' 'market_energy'
 'forming' 'neutral']


In [ ]:

# ── 2. a.emotion_secondary
print("\n emotion_emotion_category value counts")
print(df["emotion_category"].unique())
print(df["emotion_category"].nunique())
print(df["emotion_category"].value_counts().to_string())

# b.emotion_secondary
print("\n emotion_secondary value counts (incl. nulls) ──")
print(df["emotion_secondary"].unique())
print(df["emotion_secondary"].nunique())
print(df["emotion_secondary"].value_counts(dropna=False).to_string())


 emotion_emotion_category value counts
['neutral' 'joy' 'celebration' 'sarcasm' 'hustle_fatigue' 'anger'
 'prayer_gratitude' 'forming' 'suspicion' 'shock' 'pride' 'market_energy'
 'craving' 'contempt' 'betrayal' 'hustle_energy']
16
emotion_category
neutral             91
hustle_fatigue      61
pride               45
anger               39
prayer_gratitude    34
joy                 30
sarcasm             28
celebration         25
betrayal            21
suspicion           20
shock               20
market_energy       20
craving             20
forming             17
hustle_energy       15
contempt            14

 emotion_secondary value counts (incl. nulls) ──
[nan 'betrayal' 'anger' 'shock' 'suspicion' 'hustle_fatigue' 'pride'
 'prayer_gratitude' 'joy' 'celebration' 'craving' 'market_energy'
 'forming' 'neutral']
13
emotion_secondary
NaN                 353
hustle_fatigue       29
pride                27
anger                20
suspicion            19
celebration          13
betrayal  

In [ ]:
#3. Emotion co-occurrence map 
print("\n── emotion_secondary value counts (incl. nulls) ──")
print(df["emotion_secondary"].value_counts(dropna=False).to_string())


print("\n  Primary → Secondary co-occurrences")
has_secondary = df[df["emotion_secondary"].notna()]
#print(has_secondary)
has_secondary=has_secondary.groupby("emotion_category")["emotion_secondary"]
co = (has_secondary.apply(lambda x: sorted(x.str.strip().str.lower().unique().tolist())))
for primary, secondaries in co.items():
    print(f"  {primary:<20} → {secondaries}")


── emotion_secondary value counts (incl. nulls) ──
emotion_secondary
NaN                 353
hustle_fatigue       29
pride                27
anger                20
suspicion            19
celebration          13
betrayal              9
prayer_gratitude      7
shock                 6
joy                   5
forming               5
craving               4
market_energy         2
neutral               1

  Primary → Secondary co-occurrences
  anger                → ['betrayal', 'hustle_fatigue', 'pride', 'shock', 'suspicion']
  betrayal             → ['anger', 'forming', 'hustle_fatigue', 'suspicion']
  celebration          → ['hustle_fatigue', 'prayer_gratitude', 'pride']
  contempt             → ['anger', 'neutral', 'pride', 'suspicion']
  craving              → ['hustle_fatigue', 'joy', 'prayer_gratitude', 'pride']
  forming              → ['anger', 'hustle_fatigue', 'pride', 'suspicion']
  hustle_energy        → ['celebration', 'craving', 'hustle_fatigue', 'joy', 'pride']
  hustle_f

In [28]:
# 4. Sentiment distribution 
print("\n── Sentiment distribution ──")
print(df["sentiment"].value_counts().to_string())

# 5. Sarcasm + prosody joint distribution 
print("\n── Sarcasm × prosody cross-tab ──")
print(pd.crosstab(df["sarcasm_flag"], df["prosody_match"]))

# 6. How often does sarcasm_flag=yes BUT prosody=matched? 
conflict = df[(df["sarcasm_flag"] == "yes") & (df["prosody_match"] == "matched")]



── Sentiment distribution ──
sentiment
negative    263
positive    160
neutral      77

── Sarcasm × prosody cross-tab ──
prosody_match  contradicts  matched
sarcasm_flag                       
no                       0      461
yes                     13       26


In [46]:
# ── 8. Class imbalance warning ───────────────────────────────────────────
print("\n── Emotion label frequency (primary) ──")
counts = df["emotion_category"].str.strip().str.lower().value_counts()
total = len(df)
for emotion, count in counts.items():
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    warn = "  ← rare (<3%)" if pct < 3 else ""
    print(f"  {emotion:<20} {count:>4}  {pct:5.1f}%  {bar}{warn}")




── Emotion label frequency (primary) ──
  neutral                91   18.2%  █████████
  hustle_fatigue         61   12.2%  ██████
  pride                  45    9.0%  ████
  anger                  39    7.8%  ███
  prayer_gratitude       34    6.8%  ███
  joy                    30    6.0%  ███
  sarcasm                28    5.6%  ██
  celebration            25    5.0%  ██
  betrayal               21    4.2%  ██
  suspicion              20    4.0%  ██
  shock                  20    4.0%  ██
  market_energy          20    4.0%  ██
  craving                20    4.0%  ██
  forming                17    3.4%  █
  hustle_energy          15    3.0%  █
  contempt               14    2.8%  █  ← rare (<3%)


In [47]:
# ── 9. Intensity distribution ────────────────────────────────────────────
print("\n── Intensity distribution ──")
print(df["intensity"].value_counts().sort_index().to_string())


── Intensity distribution ──
intensity
1    109
2    245
3    132
4     11
5      3


In [48]:
# ── 10. Register distribution ─────────────────────────────────────────────
print("\n── Register distribution ──")
print(df["register"].value_counts().to_string())


── Register distribution ──
register
casual        365
street        128
proverbial      7
